# Notebook for testing downhole collection construction

This notebook is for testing the creation of a DownholeCollection (intermediary object) and conversion to an Evo Downhole Collection.

A cell at the end of this notebook is available to publish the result if desired.

This notebook creates a fully synthetic downhole collection.

In [ ]:
import os
from evo.notebooks import ServiceManagerWidget

# Credentials can be provided from .env or filled into second params below.
# Use `uv run --env-file .env` to load environment variables from .env file.
client_id = os.getenv("EVO_CLIENT_ID", "")
base_uri = os.getenv("EVO_BASE_URI", "")
discovery_url = os.getenv("EVO_DISCOVERY_URL", "")

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id, base_uri=base_uri, discovery_url=discovery_url
).login()

In [ ]:
from evo.data_converters.common import create_evo_object_service_and_data_client

object_service_client, data_client = create_evo_object_service_and_data_client(service_manager_widget=manager)

In [ ]:
import numpy as np
import pandas as pd
import random
from decimal import Decimal
from uuid import uuid4
from pprint import pp
from datetime import datetime, timezone
from evo.data_converters.common.objects import DownholeCollectionToGeoscienceObject
from evo.data_converters.common.objects.downhole_collection import ColumnMapping, DownholeCollection
from evo.data_converters.common.objects.downhole_collection.tables import MeasurementTableFactory
from evo.data_converters.common.objects.downhole_collection.hole_collars import HOLE_COLLARS_SCHEMA, HoleCollars

# Generate hole collars
collars_df = pd.DataFrame(
    columns=["hole_index", "hole_id", "x", "y", "z", "final_depth", "SOME_ATTR"],
    data=[
        (1, "hole-1", 100, 200, 0.0, 20.0, "Foo"),
        (2, "hole-2", 103, 206, 0.2, 20.0, "Bar"),
        (3, "hole-3", 99, 199, -0.1, 20.0, "Baz"),
    ],
).astype(HOLE_COLLARS_SCHEMA)
collars = HoleCollars(df=collars_df, nan_values_by_column={})

downhole_collection = DownholeCollection(collars=collars, name="Synthetic DHC")

# Generate distance table
sample_cat: list[str] = ["cat1", "cat2", "cat3", "cat4", "cat5", None]

distance_table_data = {
    # Hole index will come from LOCA_ID and mapped to the index of the hole_id in the collars df using 1-based arrays
    "hole_index": [],
    "penetration_length": [],
    "resistance": [],
    "BOOL_ATTR": [],
    "STRING_ATTR": [],
    "DATE_ATTR": [],
    "CAT_ATTR": [],
}

# each hole has measurements from 0 to 20m at 0.1m intervals
for hole_id in range(1, len(collars_df) + 1):
    cur_depth = Decimal("0.0")

    while cur_depth < 20:
        distance_table_data["hole_index"].append(hole_id)
        distance_table_data["penetration_length"].append(float(cur_depth))

        if cur_depth == Decimal("0.9"):
            # Stick some regular NaNs here
            distance_table_data["resistance"].append(np.nan)
            distance_table_data["BOOL_ATTR"].append(None)
            distance_table_data["STRING_ATTR"].append(None)
            distance_table_data["DATE_ATTR"].append(None)
            distance_table_data["CAT_ATTR"].append(None)
        elif cur_depth == Decimal("1.9"):
            # Stick some NaN placeholders here
            distance_table_data["resistance"].append(9999.99)
            distance_table_data["BOOL_ATTR"].append(None)
            distance_table_data["STRING_ATTR"].append("")
            distance_table_data["DATE_ATTR"].append(pd.NA)
            distance_table_data["CAT_ATTR"].append(pd.NA)
        else:
            distance_table_data["resistance"].append(random.uniform(0.1, 1.0))
            distance_table_data["BOOL_ATTR"].append(random.choice([True, False]))
            distance_table_data["STRING_ATTR"].append(uuid4())
            distance_table_data["DATE_ATTR"].append(datetime.now(tz=timezone.utc))
            distance_table_data["CAT_ATTR"].append(random.choice(sample_cat))

        cur_depth += Decimal("0.1")

distance_table = pd.DataFrame(distance_table_data).astype({"CAT_ATTR": "category"})
pp(distance_table.iloc[8:15])
downhole_collection.add_measurement_table(
    MeasurementTableFactory.create(
        df=distance_table,
        column_mapping=ColumnMapping(DEPTH_COLUMNS=["penetration_length"]),
        nan_values_by_column={"resistance": [9999.99], "DATE_ATTR": [-1]},
    )
)

# Generate a somewhat complex interval table
lithologies: list[str] = ["sandstone", "limestone", "shale", "granite", "basalt", "mudstone", "conglomerate"]

interval_table_data = {
    # Hole index will come from LOCA_ID and mapped to the index of the hole_id in the collars df using 1-based arrays
    "hole_index": [],
    "GEOL_TOP": [],
    "GEOL_BASE": [],
    "GEOL_DENSITY": [],
    "GEOL_LITHOLOGY": [],
}

# each hole has intervals from 0 to 20m (based on sample data in ./data/input)
for hole_id in range(1, len(collars_df) + 1):
    cur_depth = 0

    while cur_depth < 20:
        interval_size = random.uniform(1.5, 4.0)

        # Make sure we don't exceed 20m
        if cur_depth + interval_size > 20:
            interval_size = 20 - cur_depth

        density = random.uniform(1.2, 2.6)

        interval_table_data["hole_index"].append(hole_id)
        interval_table_data["GEOL_TOP"].append(cur_depth)
        interval_table_data["GEOL_BASE"].append(cur_depth + interval_size)
        interval_table_data["GEOL_DENSITY"].append(density)
        interval_table_data["GEOL_LITHOLOGY"].append(random.choice(lithologies))

        cur_depth += interval_size

interval_table = pd.DataFrame(interval_table_data)

# create categorical attribute
interval_table["GEOL_LITHOLOGY"] = interval_table["GEOL_LITHOLOGY"].astype("category")

display(interval_table)

downhole_collection.add_measurement_table(
    MeasurementTableFactory.create(
        df=interval_table, column_mapping=ColumnMapping(FROM_COLUMNS=["GEOL_TOP"], TO_COLUMNS=["GEOL_BASE"])
    )
)

converter = DownholeCollectionToGeoscienceObject(dhc=downhole_collection, data_client=data_client)
geoscience_object = converter.convert()

pp(geoscience_object)

Optionally, you can publish the constructed downhole collection by running the following

In [ ]:
from evo.data_converters.common import publish_geoscience_objects

result = publish_geoscience_objects(
    object_models=[geoscience_object],
    object_service_client=object_service_client,
    data_client=data_client,
    path_prefix="dhc-notebook",
    overwrite_existing_objects=True,
)

pp(result)